In [1]:
import os, shutil, random, time, yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from ultralytics import YOLO
import ultralytics

In [2]:
DATASET_ROOT   = "datasets/qr_obb"
CLASS_NAME     = "qr_code"
AUGMENT_FACTOR = 6
RUN_NAME       = "run_v1"
PROJECT        = "QR_OBB_Training"
BEST_PT        = f"./runs/obb/{PROJECT}/{RUN_NAME}/weights/best.pt"
BEST_ONNX      = f"./runs/obb/{PROJECT}/{RUN_NAME}/weights/best.onnx"

In [3]:
def setup_dirs(force_clear=True):
    if force_clear and os.path.exists(DATASET_ROOT):
        shutil.rmtree(DATASET_ROOT)
    for split in ['train', 'val']:
        for sub in ['images', 'labels']:
            Path(os.path.join(DATASET_ROOT, split, sub)).mkdir(parents=True, exist_ok=True)
    Path(os.path.join(DATASET_ROOT, "debug")).mkdir(parents=True, exist_ok=True)
    print(f"[*] Đã khởi tạo thư mục: {DATASET_ROOT}")

In [4]:
setup_dirs(force_clear=True)

[*] Đã khởi tạo thư mục: datasets/qr_obb


In [5]:
def transform_point(x, y, M):
    return M[0,0]*x + M[0,1]*y + M[0,2], M[1,0]*x + M[1,1]*y + M[1,2]

def is_box_inside_image(obb_pts, img_w, img_h):
    return all(0 <= x < img_w and 0 <= y < img_h for x, y in obb_pts)

In [ ]:
def apply_photometric_augmentation(img):
    aug = img.copy().astype(np.float32)

    if random.random() < 0.7:                          # Brightness
        aug *= random.uniform(0.3, 2.2)
    if random.random() < 0.6:                          # Contrast
        f = random.uniform(0.4, 2.5)
        aug = (aug - aug.mean()) * f + aug.mean()
    if random.random() < 0.5:                          # Gamma
        aug = np.power(np.clip(aug/255.0, 0, 1), random.uniform(0.35, 2.8)) * 255.0

    aug = np.clip(aug, 0, 255).astype(np.uint8)

    if random.random() < 0.5:                          # Gaussian Blur
        k = random.choice([3, 5, 7, 9, 11])
        aug = cv2.GaussianBlur(aug, (k, k), 0)

    if random.random() < 0.35:                         # Motion Blur
        k = random.choice([5, 7, 9, 13, 17])
        kernel = np.zeros((k, k), np.float32)
        kernel[k//2, :] = 1.0 / k
        rm = cv2.getRotationMatrix2D((k/2, k/2), random.uniform(0, 180), 1)
        kernel = cv2.warpAffine(kernel, rm, (k, k))
        kernel /= kernel.sum() + 1e-6
        aug = cv2.filter2D(aug, -1, kernel)

    if random.random() < 0.6:                          # Gaussian Noise
        noise = np.random.normal(0, random.uniform(8, 55), aug.shape).astype(np.float32)
        aug = np.clip(aug.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if random.random() < 0.35:                         # Salt & Pepper
        n = int(random.uniform(0.01, 0.06) * aug.size * 0.5)
        out = aug.copy()
        out[np.random.randint(0, aug.shape[0], n), np.random.randint(0, aug.shape[1], n)] = 255
        out[np.random.randint(0, aug.shape[0], n), np.random.randint(0, aug.shape[1], n)] = 0
        aug = out

    if random.random() < 0.45:                         # JPEG artifact
        q = random.randint(5, 40)
        _, enc = cv2.imencode('.jpg', aug, [cv2.IMWRITE_JPEG_QUALITY, q])
        aug = cv2.imdecode(enc, cv2.IMREAD_COLOR)

    if random.random() < 0.5:                          # Color Jitter HSV
        hsv = cv2.cvtColor(aug, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = (hsv[:,:,0] + random.uniform(-25, 25)) % 180
        hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.4, 1.8), 0, 255)
        hsv[:,:,2] = np.clip(hsv[:,:,2] * random.uniform(0.4, 1.8), 0, 255)
        aug = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    if random.random() < 0.2:                          # Grayscale
        aug = cv2.cvtColor(cv2.cvtColor(aug, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)

    if random.random() < 0.4:                          # Shadow patch
        h_i, w_i = aug.shape[:2]
        shadow = aug.copy().astype(np.float32)
        pts_s = np.array([[0, random.randint(0,h_i)],
                          [w_i, random.randint(0,h_i)],
                          [w_i, h_i], [0, h_i]], np.int32)
        mask = np.zeros((h_i, w_i), np.uint8)
        cv2.fillPoly(mask, [pts_s], 1)
        shadow[mask == 1] *= random.uniform(0.15, 0.55)
        aug = np.clip(shadow, 0, 255).astype(np.uint8)

    return aug

In [7]:
def augment_and_save_sample(img, image_id, group, split, suffix,
                            angle=0, dx=0, dy=0, debug=False):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    M[0,2] += dx
    M[1,2] += dy
    img_aug = cv2.warpAffine(img, M, (w, h), borderValue=(0,0,0))

    if split == 'train' and suffix != '_orig':
        img_aug = apply_photometric_augmentation(img_aug)

    img_debug = img_aug.copy() if debug else None
    valid_lines = []

    if group is not None:
        for _, row in group.iterrows():
            if pd.isna(row['x0']):
                continue
            pts = [[*transform_point(row[f'x{i}'], row[f'y{i}'], M)] for i in range(4)]
            if not is_box_inside_image(pts, w, h):
                continue
            norm = []
            for p in pts:
                norm.extend([p[0]/w, p[1]/h])
            valid_lines.append(f"0 {' '.join(f'{v:.6f}' for v in norm)}\n")
            if debug:
                cv2.polylines(img_debug,
                              [np.array(pts, np.int32).reshape(-1,1,2)], True, (0,255,0), 2)
                for i, p in enumerate(pts):
                    cv2.putText(img_debug, str(i), (int(p[0]), int(p[1])),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    if split == 'train' and not valid_lines:
        return False

    new_id = f"{image_id}{suffix}"
    cv2.imwrite(f"{DATASET_ROOT}/{split}/images/{new_id}.jpg", img_aug)
    with open(f"{DATASET_ROOT}/{split}/labels/{new_id}.txt", 'w') as f:
        f.writelines(valid_lines)
    if debug:
        cv2.imwrite(f"{DATASET_ROOT}/debug/{new_id}_debug.jpg", img_debug)
    return True

In [8]:
def process_dataset(csv_path, src_img_dir):
    df = pd.read_csv(csv_path)
    all_images = [f.replace('.jpg','') for f in os.listdir(src_img_dir) if f.endswith('.jpg')]
    random.seed(42)
    random.shuffle(all_images)

    split_idx = int(len(all_images) * 0.8)
    train_imgs, val_imgs = all_images[:split_idx], all_images[split_idx:]
    grouped = df.groupby('image_id')

    print(f"[*] VAL: {len(val_imgs)} ảnh...")
    for img_id in val_imgs:
        img   = cv2.imread(os.path.join(src_img_dir, f"{img_id}.jpg"))
        group = grouped.get_group(img_id) if img_id in grouped.groups else None
        augment_and_save_sample(img, img_id, group, 'val', "")

    print(f"[*] TRAIN: {len(train_imgs)} ảnh gốc × {AUGMENT_FACTOR}...")
    count = 0
    for idx, img_id in enumerate(train_imgs):
        img = cv2.imread(os.path.join(src_img_dir, f"{img_id}.jpg"))
        if img is None: continue
        group    = grouped.get_group(img_id) if img_id in grouped.groups else None
        h, w     = img.shape[:2]
        is_debug = idx < 3

        augment_and_save_sample(img, img_id, group, 'train', "_orig", debug=is_debug)
        count += 1
        for i in range(1, AUGMENT_FACTOR):
            ok = augment_and_save_sample(
                img, img_id, group, 'train', f"_aug{i}",
                angle=random.uniform(0, 360),
                dx=random.uniform(-w*0.2, w*0.2),
                dy=random.uniform(-h*0.2, h*0.2),
                debug=is_debug
            )
            if ok: count += 1

    print(f"[*] Hoàn tất! Tổng ảnh train: {count}")


In [ ]:
process_dataset('qr/output_train.csv', 'qr/train')

[*] VAL: 217 ảnh...
[*] TRAIN: 866 ảnh gốc × 6...


In [ ]:
def create_obb_yaml():
    yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump({
            'path' : os.path.abspath(DATASET_ROOT),
            'train': 'train/images',
            'val'  : 'val/images',
            'names': {0: CLASS_NAME}
        }, f, default_flow_style=False, sort_keys=False)
    assert os.path.exists(yaml_path), "Tạo yaml thất bại!"
    print(f"[*] Đã tạo & xác nhận: {yaml_path}")

In [ ]:
create_obb_yaml()

[*] Đã tạo & xác nhận: datasets/qr_obb/data.yaml


In [ ]:
ultralytics.checks()

Ultralytics 8.4.33 🚀 Python-3.11.15 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5764MiB)
Setup complete ✅ (16 CPUs, 30.1 GB RAM, 18.4/109.4 GB disk)


In [ ]:
model = YOLO('yolov8n-obb.pt')
model.train(
        data       = f'{DATASET_ROOT}/data.yaml',
        epochs     = 100, imgsz=640, batch=42,
        patience   = 10,  device=0,
        project    = PROJECT, name=RUN_NAME,
        save       = True, verbose=True,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        degrees=0.0, translate=0.0, scale=0.0,
        shear=0.0, flipud=0.0, fliplr=0.0,
        mosaic=0.0, mixup=0.0, copy_paste=0.0,
    )
print(f"[*] Huấn luyện xong → {BEST_PT}")

New https://pypi.org/project/ultralytics/8.4.39 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.11.15 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5764MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/qr_obb/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name

/home/sanng/miniconda3/envs/vision_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1890.8±907.4 MB/s, size: 103.3 KB)
val: Scanning /run/media/sanng/New Volume/Window/Python/QR_DETECTION/datasets/qr_obb/val/labels... 217 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 217/217 2.7Kit/s 0.1s
val: New cache created: /run/media/sanng/New Volume/Window/Python/QR_DETECTION/datasets/qr_obb/val/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 63 weight(decay=0.0), 73 weight(decay=0.0005), 72 bias(decay=0.0)
Plotting labels to /run/media/sanng/New Volume/Window/Python/QR_DETECTION/runs/obb/QR_OBB_Training/run_v1/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /run/media/sanng/New Volume/Window/Python/QR_DETECTION/runs/obb/QR_OBB_Training/run_v1
Starting training for 200 epochs...

      Epoch    GPU_mem   bo

In [ ]:
def plot_training_metrics(run_name=RUN_NAME, project=PROJECT):
    csv_path = './runs/obb'/ Path(project) / run_name / 'results.csv'
    if not csv_path.exists():
        print(f"[!] Không tìm thấy: {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    epochs = df['epoch'].values
    # ── Phải lấy lr_cols ở đây để dùng cho cả Figure 3 lẫn Dashboard ──
    lr_cols = [c for c in df.columns if c.startswith('lr/')]

    print(f"[*] Loaded {len(df)} epochs | Cột: {list(df.columns)}")

    C_TRAIN = '#2196F3'; C_VAL = '#F44336'
    C_METRIC = '#4CAF50'; C_LR = '#FF9800'; FILL = 0.12

    def safe(col):
        return df[col].values if col in df.columns else None

    # ── Figure 1: Loss ────────────────────────────────────────
    fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
    fig1.suptitle('Training vs Validation Loss', fontsize=16, fontweight='bold')
    for ax, (title, tc, vc) in zip(axes1, [
        ('Box Loss', 'train/box_loss', 'val/box_loss'),
        ('Cls Loss', 'train/cls_loss', 'val/cls_loss'),
        ('DFL Loss', 'train/dfl_loss', 'val/dfl_loss'),
    ]):
        tv, vv = safe(tc), safe(vc)
        if tv is not None:
            ax.plot(epochs, tv, color=C_TRAIN, lw=2, label='Train')
            ax.fill_between(epochs, tv, alpha=FILL, color=C_TRAIN)
        if vv is not None:
            ax.plot(epochs, vv, color=C_VAL, lw=2, label='Val')
            ax.fill_between(epochs, vv, alpha=FILL, color=C_VAL)
            bi = np.argmin(vv)
            ax.axvline(epochs[bi], color=C_VAL, lw=1, ls='--', alpha=0.6)
            ax.scatter(epochs[bi], vv[bi], color=C_VAL, s=60, zorder=5)
            ax.annotate(f'min={vv[bi]:.4f}\n@ep{epochs[bi]}',
                        xy=(epochs[bi], vv[bi]), xytext=(8,10),
                        textcoords='offset points', fontsize=8, color=C_VAL)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        ax.set_xlim(epochs[0], epochs[-1])
    fig1.tight_layout()
    fig1.savefig(f'./runs/obb/{project}/{run_name}/plot_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 2: Metrics ─────────────────────────────────────
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    fig2.suptitle('Metrics: mAP / Precision / Recall', fontsize=16, fontweight='bold')
    for ax, (title, col, color) in zip(axes2.flatten(), [
        ('mAP@50',    'metrics/mAP50(B)',    C_METRIC),
        ('mAP@50:95', 'metrics/mAP50-95(B)', '#9C27B0'),
        ('Precision', 'metrics/precision(B)','#00BCD4'),
        ('Recall',    'metrics/recall(B)',    '#FF5722'),
    ]):
        vals = safe(col)
        if vals is None:
            ax.text(0.5, 0.5, f'Không có\n{col}', ha='center', va='center',
                    transform=ax.transAxes); ax.set_title(title); continue
        ax.plot(epochs, vals, color=color, lw=2.5)
        ax.fill_between(epochs, vals, alpha=FILL, color=color)
        bi = np.argmax(vals)
        ax.axvline(epochs[bi], color=color, lw=1, ls='--', alpha=0.6)
        ax.scatter(epochs[bi], vals[bi], color=color, s=70, zorder=5)
        ax.annotate(f'max={vals[bi]:.4f}\n@ep{epochs[bi]}',
                    xy=(epochs[bi], vals[bi]), xytext=(8,-20),
                    textcoords='offset points', fontsize=8.5, color=color)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(title)
        ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
        ax.set_xlim(epochs[0], epochs[-1])
    fig2.tight_layout()
    fig2.savefig(f'./runs/obb/{project}/{run_name}/plot_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 3: Learning Rate ───────────────────────────────
    if lr_cols:
        fig3, ax3 = plt.subplots(figsize=(10, 4))
        fig3.suptitle('Learning Rate Schedule', fontsize=16, fontweight='bold')
        for i, col in enumerate(lr_cols):
            ax3.plot(epochs, df[col].values, lw=2,
                     color=[C_LR,'#795548','#607D8B'][i % 3],
                     label=col.replace('lr/','pg'))
        ax3.set_xlabel('Epoch'); ax3.set_ylabel('LR')
        ax3.set_yscale('log'); ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3, which='both')
        ax3.set_xlim(epochs[0], epochs[-1])
        fig3.tight_layout()
        fig3.savefig(f'./runs/obb/{project}/{run_name}/plot_lr.png', dpi=150, bbox_inches='tight')
        plt.show()

    # ── Figure 4: Dashboard ───────────────────────────────────
    fig4 = plt.figure(figsize=(20, 12))
    fig4.suptitle(f'Training Dashboard — {project}/{run_name}',
                  fontsize=17, fontweight='bold')
    gs = gridspec.GridSpec(3, 4, figure=fig4, hspace=0.45, wspace=0.35)

    specs = [
        (gs[0,0],'Box Loss',  'train/box_loss',        'val/box_loss',         'loss'),
        (gs[0,1],'Cls Loss',  'train/cls_loss',         'val/cls_loss',         'loss'),
        (gs[0,2],'DFL Loss',  'train/dfl_loss',         'val/dfl_loss',         'loss'),
        (gs[0,3],'mAP@50',    'metrics/mAP50(B)',       None,                   'metric'),
        (gs[1,0],'mAP@50:95', 'metrics/mAP50-95(B)',    None,                   'metric'),
        (gs[1,1],'Precision', 'metrics/precision(B)',   None,                   'metric'),
        (gs[1,2],'Recall',    'metrics/recall(B)',      None,                   'metric'),
        (gs[1,3],'LR',        lr_cols[0] if lr_cols else None, None,            'lr'),
    ]
    cmap = {'loss':(C_TRAIN,C_VAL), 'metric':(C_METRIC,None), 'lr':(C_LR,None)}

    for pos, title, ca_col, cb_col, kind in specs:
        ax = fig4.add_subplot(pos)
        if ca_col is None: ax.axis('off'); continue
        ca, cb = cmap[kind]
        va, vb = safe(ca_col), (safe(cb_col) if cb_col else None)
        if va is not None:
            ax.plot(epochs, va, color=ca, lw=1.8,
                    label='Train' if kind=='loss' else title)
            ax.fill_between(epochs, va, alpha=FILL, color=ca)
        if vb is not None:
            ax.plot(epochs, vb, color=cb, lw=1.8, label='Val')
            ax.fill_between(epochs, vb, alpha=FILL, color=cb)
        if kind == 'lr': ax.set_yscale('log')
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8)
        ax.grid(True, alpha=0.25); ax.tick_params(labelsize=7)
        ax.set_xlim(epochs[0], epochs[-1])
        if kind in ('loss','metric') and va is not None and vb is not None:
            ax.legend(fontsize=7)

    # Bảng tóm tắt
    ax_tbl = fig4.add_subplot(gs[2,:])
    ax_tbl.axis('off')
    rows, heads = [], ['Chỉ số','Epoch tốt nhất','Giá trị tốt nhất','Epoch cuối','Giá trị cuối']
    for col, label in [
        ('metrics/mAP50(B)',     'mAP@50'),
        ('metrics/mAP50-95(B)', 'mAP@50:95'),
        ('metrics/precision(B)','Precision'),
        ('metrics/recall(B)',   'Recall'),
        ('val/box_loss',        'Val Box Loss'),
        ('val/cls_loss',        'Val Cls Loss'),
        ('val/dfl_loss',        'Val DFL Loss'),
    ]:
        v = safe(col)
        if v is None: continue
        bi = np.argmin(v) if 'loss' in col else np.argmax(v)
        rows.append([label, str(epochs[bi]), f"{v[bi]:.5f}", str(epochs[-1]), f"{v[-1]:.5f}"])

    if rows:
        tbl = ax_tbl.table(cellText=rows, colLabels=heads, cellLoc='center', loc='center')
        tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 1.6)
        for j in range(len(heads)):
            tbl[0,j].set_facecolor('#1976D2')
            tbl[0,j].set_text_props(color='white', fontweight='bold')
        for i in range(1, len(rows)+1):
            for j in range(len(heads)):
                tbl[i,j].set_facecolor('#F5F5F5' if i%2==0 else 'white')

    fig4.savefig(f'./runs/obb/{project}/{run_name}/plot_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[*] Đã lưu 4 biểu đồ → {project}/{run_name}/")


In [ ]:
plot_training_metrics()

[*] Loaded 62 epochs | Cột: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'train/angle_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'val/angle_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2', 'lr/pg3', 'lr/pg4', 'lr/pg5', 'lr/pg6', 'lr/pg7']


<Figure size 1000x400 with 1 Axes>

<Figure size 1800x500 with 3 Axes>

<Figure size 1400x1000 with 4 Axes>

<Figure size 1000x400 with 1 Axes>

<Figure size 2000x1200 with 9 Axes>

[*] Đã lưu 4 biểu đồ → QR_OBB_Training/run_v1/


In [ ]:
assert os.path.exists(BEST_PT), f"❌ Không tìm thấy {BEST_PT}"
YOLO(BEST_PT).export(format='onnx', opset=12)
print(f"[*] Đã export → {BEST_ONNX}")


Ultralytics 8.4.33 🚀 Python-3.11.15 torch-2.4.0 CPU (AMD Ryzen 7 7735HS with Radeon Graphics)
YOLOv8n-obb summary (fused): 82 layers, 3,077,414 parameters, 0 gradients, 8.3 GFLOPs

PyTorch: starting from 'runs/obb/QR_OBB_Training/run_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.6 MB)

ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 0.6s, saved as 'runs/obb/QR_OBB_Training/run_v1/weights/best.onnx' (11.9 MB)

Export complete (0.8s)
Results saved to /run/media/sanng/New Volume/Window/Python/QR_DETECTION/runs/obb/QR_OBB_Training/run_v1/weights
Predict:         yolo predict task=obb model=runs/obb/QR_OBB_Training/run_v1/weights/best.onnx imgsz=640 
Validate:        yolo val task=obb model=runs/obb/QR_OBB_Training/run_v1/weights/best.onnx imgsz=640 data=datasets/qr_obb/data.yaml  
Visualize:       https://netron.app
[*] Đã export → QR_OBB_Training/run_v1/weights/best.onnx


In [ ]:
def order_points_clockwise(pts):
    s, diff = pts.sum(axis=1), np.diff(pts, axis=1)
    rect = np.zeros((4,2), dtype="float32")
    rect[0]=pts[np.argmin(s)]; rect[2]=pts[np.argmax(s)]
    rect[1]=pts[np.argmin(diff)]; rect[3]=pts[np.argmax(diff)]
    return rect

def run_camera_test(model_path, camera_id=0, conf=0.4, imgsz=640):
    model = YOLO(model_path)
    cap   = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print(f"[!] Không mở được camera {camera_id}"); return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[*] Q/ESC: thoát | +/-: chỉnh conf")
    fps_history, current_conf = [], conf

    while True:
        t0 = time.time()
        ret, frame = cap.read()
        if not ret: break

        res  = model.predict(source=frame, device='cpu',
                             imgsz=imgsz, conf=current_conf, verbose=False)
        obb  = res[0].obb
        cnt  = 0
        colors = [(0,255,0),(0,165,255),(255,0,255),(0,255,255)]

        if obb is not None and len(obb) > 0:
            pts_all = obb.xyxyxyxy.cpu().numpy()
            confs   = obb.conf.cpu().numpy()
            cnt     = len(pts_all)
            for i, (pts, cf) in enumerate(zip(pts_all, confs)):
                ordered  = order_points_clockwise(pts)
                color    = colors[i % len(colors)]
                cv2.polylines(frame, [ordered.astype(np.int32).reshape(-1,1,2)], True, color, 2)
                for pt in ordered:
                    cv2.circle(frame, (int(pt[0]), int(pt[1])), 5, color, -1)
                cv2.putText(frame, f"QR#{i} {cf:.2f}",
                            (int(ordered[:,0].min()), max(int(ordered[:,1].min())-10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        fps = 1.0 / (time.time() - t0 + 1e-9)
        fps_history = (fps_history + [fps])[-30:]
        cv2.putText(frame,
                    f"QR:{cnt}  conf:{current_conf:.2f}  "
                    f"FPS:{sum(fps_history)/len(fps_history):.1f}  [+/-][Q]",
                    (8,27), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
        cv2.imshow('QR Detection', frame)

        key = cv2.waitKey(1) & 0xFF
        if key in [ord('q'), 27]: break
        elif key in [ord('+'), ord('=')]:
            current_conf = min(0.95, round(current_conf+0.05, 2))
            print(f"conf → {current_conf}")
        elif key == ord('-'):
            current_conf = max(0.05, round(current_conf-0.05, 2))
            print(f"conf → {current_conf}")

    cap.release(); cv2.destroyAllWindows()

In [ ]:
run_camera_test(model_path=BEST_ONNX, camera_id=0, conf=0.4)

[*] Q/ESC: thoát | +/-: chỉnh conf
Loading ./runs/obb/QR_OBB_Training/run_v1/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CPUExecutionProvider


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""
